# TP 0 — Understanding Boosting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/L2Math/blob/main/session2/tp_0_boosting.ipynb)

## Objectives
1. Understand the key difference between Bagging and Boosting
2. Learn how AdaBoost reweights samples to focus on mistakes
3. Understand Gradient Boosting as iterative residual fitting
4. Visualize decision boundaries and learning curves for both methods
5. Explore overfitting behavior in boosting algorithms

## Setup

Run the cell below to install and import the required packages.

In [ ]:
!pip install git+https://github.com/racousin/L2Math.git

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from l2math import plot_decision_boundary_2d

print("Setup complete!")

---
# Part 1: Bagging vs Boosting — Key Difference

Both **Bagging** and **Boosting** are ensemble methods that combine many weak learners. However, they work in fundamentally different ways:

| | **Bagging** (e.g., Random Forest) | **Boosting** (e.g., AdaBoost, Gradient Boosting) |
|---|---|---|
| Training | **Parallel** — each tree is trained independently | **Sequential** — each tree depends on the previous ones |
| Goal | Reduce **variance** | Reduce **bias** |
| Data sampling | Each tree sees a bootstrap sample | Each tree focuses on previously misclassified samples |
| Combination | Simple average / majority vote | Weighted combination |

### Visual Intuition

**Bagging (parallel):**
```
Data ──┬── Tree 1 ──┐
       ├── Tree 2 ──┤
       ├── Tree 3 ──┼──> Average / Vote ──> Prediction
       ├── Tree 4 ──┤
       └── Tree 5 ──┘
```

**Boosting (sequential):**
```
Data ──> Tree 1 ──> Errors ──> Tree 2 ──> Errors ──> Tree 3 ──> ... ──> Weighted Sum ──> Prediction
```

In boosting, each new tree **focuses on the mistakes** made by the ensemble so far.

---
# Part 2: AdaBoost — Weighted Samples

## Intuition

AdaBoost (**Ada**ptive **Boost**ing) works by training a sequence of weak learners (typically decision stumps). After each learner, samples that were **misclassified** receive **higher weight**, so the next learner focuses more on the hard cases.

## Algorithm (step-by-step)

Given a training set $\{(x_i, y_i)\}_{i=1}^{n}$ with $y_i \in \{-1, +1\}$:

1. **Initialize** sample weights: $w_i = \frac{1}{n}$ for all $i$

2. **For each round** $t = 1, 2, \ldots, T$:

   a. Train a weak learner $h_t$ on the weighted training data

   b. Compute the **weighted error**:
   $$\epsilon_t = \frac{\sum_{i=1}^{n} w_i \cdot \mathbb{1}[h_t(x_i) \neq y_i]}{\sum_{i=1}^{n} w_i}$$

   c. Compute the **learner weight**:
   $$\alpha_t = \frac{1}{2} \ln \frac{1 - \epsilon_t}{\epsilon_t}$$

   d. **Update sample weights**:
   $$w_i \leftarrow w_i \cdot \exp\bigl(-\alpha_t \cdot y_i \cdot h_t(x_i)\bigr)$$

   e. Normalize weights so they sum to 1

3. **Final prediction**: $H(x) = \text{sign}\!\left(\sum_{t=1}^{T} \alpha_t \cdot h_t(x)\right)$

## Visualizing Sample Weight Evolution

Let's watch how AdaBoost progressively increases the weight of misclassified samples across 4 iterations.

In [ ]:
np.random.seed(42)

# Generate data
X_moons, y_moons = make_moons(n_samples=200, noise=0.3, random_state=42)
# Convert labels to {-1, +1} for AdaBoost math
y_signed = 2 * y_moons - 1

n_samples = len(X_moons)
weights = np.ones(n_samples) / n_samples  # Step 1: uniform weights

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for t in range(4):
    ax = axes[t]

    # Train a decision stump on weighted data
    stump = DecisionTreeClassifier(max_depth=1, random_state=t)
    stump.fit(X_moons, y_signed, sample_weight=weights)
    y_pred = stump.predict(X_moons)

    # Compute weighted error
    misclassified = (y_pred != y_signed).astype(float)
    epsilon = np.dot(weights, misclassified) / np.sum(weights)
    epsilon = np.clip(epsilon, 1e-10, 1 - 1e-10)

    # Compute learner weight
    alpha = 0.5 * np.log((1 - epsilon) / epsilon)

    # Plot decision boundary using contour
    h = 0.02
    x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
    y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = stump.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.15, cmap="RdBu")
    ax.contour(xx, yy, Z, colors="k", linewidths=0.5, alpha=0.5)

    # Scatter with point sizes proportional to sample weights
    sizes = 1000 * weights / weights.max()
    sizes = np.clip(sizes, 10, 500)
    scatter = ax.scatter(
        X_moons[:, 0], X_moons[:, 1],
        c=y_moons, cmap="bwr", s=sizes,
        edgecolors="k", linewidths=0.5, alpha=0.7,
    )
    ax.set_title(f"Iteration {t+1}:  $\\alpha$={alpha:.2f},  $\\epsilon$={epsilon:.3f}", fontsize=12)
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")

    # Update weights
    weights = weights * np.exp(-alpha * y_signed * y_pred)
    weights = weights / np.sum(weights)

fig.suptitle("AdaBoost: Sample Weights Across Iterations\n(point size = sample weight)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

Notice how after each iteration the misclassified points (those on the wrong side of the decision stump) **grow larger** — they receive more weight so the next stump focuses on getting them right.

---
## AdaBoost Decision Boundary Evolution

As we add more weak learners, the combined ensemble creates increasingly complex decision boundaries.

In [ ]:
np.random.seed(42)

X_moons, y_moons = make_moons(n_samples=200, noise=0.3, random_state=42)

n_estimators_list = [1, 5, 20, 100]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for i, n_est in enumerate(n_estimators_list):
    model = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=n_est,
        random_state=42,
        algorithm="SAMME",
    )
    model.fit(X_moons, y_moons)
    acc = accuracy_score(y_moons, model.predict(X_moons))
    plot_decision_boundary_2d(
        model, X_moons, y_moons,
        title=f"AdaBoost — {n_est} estimators (train acc={acc:.2f})",
        ax=axes[i],
    )

plt.tight_layout()
plt.show()

With only 1 stump the boundary is a simple line. As the number of stumps grows, AdaBoost carves out a more refined boundary that correctly classifies the crescent shapes.

---
## AdaBoost Learning Curves

Let's track train and test accuracy as we increase the number of estimators.

In [ ]:
np.random.seed(42)

X_moons, y_moons = make_moons(n_samples=300, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X_moons, y_moons, test_size=0.3, random_state=42
)

n_range = np.arange(1, 201)
train_acc = []
test_acc = []

for n in n_range:
    model = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=n,
        random_state=42,
        algorithm="SAMME",
    )
    model.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, model.predict(X_train)))
    test_acc.append(accuracy_score(y_test, model.predict(X_test)))

plt.figure(figsize=(10, 5))
plt.plot(n_range, train_acc, label="Train accuracy", linewidth=2)
plt.plot(n_range, test_acc, label="Test accuracy", linewidth=2)
plt.xlabel("Number of estimators", fontsize=12)
plt.ylabel("Accuracy", fontsize=12)
plt.title("AdaBoost — Learning Curves", fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
# Part 3: Gradient Boosting — Fitting Residuals

## Intuition

Gradient Boosting takes a different approach from AdaBoost. Instead of reweighting samples, each new tree is trained to predict the **residual errors** (mistakes) of the current ensemble.

The ensemble prediction is built **additively**:

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

where:
- $F_{m-1}(x)$ is the prediction of the ensemble after $m-1$ trees
- $h_m(x)$ is the new tree fitted to the **residuals** $r_i = y_i - F_{m-1}(x_i)$
- $\eta$ is the **learning rate** (shrinkage parameter)

### Connection to Gradient Descent in Function Space

For squared loss, the negative gradient of the loss with respect to the prediction is exactly the residual:

$$-\frac{\partial}{\partial F} \frac{1}{2}(y - F)^2 = y - F = \text{residual}$$

So fitting a tree to residuals is equivalent to taking a **gradient descent step in function space**. This is why it is called *Gradient* Boosting.

## Manual Gradient Boosting on 1D Regression

Let's build a gradient boosting regressor from scratch to see how each tree corrects the previous ensemble's errors.

In [ ]:
np.random.seed(42)

# Generate noisy sine data
X_reg = np.sort(np.random.uniform(0, 2 * np.pi, 200)).reshape(-1, 1)
y_reg = np.sin(X_reg).ravel() + 0.2 * np.random.randn(200)

# Manual gradient boosting
learning_rate = 0.3
prediction = np.zeros_like(y_reg)  # F_0(x) = 0
trees = []
snapshots = [0, 1, 3, 10]  # after 0, 1, 3, 10 trees
snapshot_preds = {}
snapshot_residuals = {}

# Store initial state
snapshot_preds[0] = prediction.copy()
snapshot_residuals[0] = y_reg - prediction

for m in range(1, 11):
    residuals = y_reg - prediction
    tree = DecisionTreeRegressor(max_depth=2, random_state=42)
    tree.fit(X_reg, residuals)
    prediction = prediction + learning_rate * tree.predict(X_reg)
    trees.append(tree)

    if m in snapshots:
        snapshot_preds[m] = prediction.copy()
        snapshot_residuals[m] = y_reg - prediction

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()
titles = ["Before boosting (0 trees)", "After 1 tree", "After 3 trees", "After 10 trees"]

for idx, m in enumerate(snapshots):
    ax = axes[idx]
    ax.scatter(X_reg, y_reg, alpha=0.3, s=15, color="gray", label="True data")
    sort_idx = np.argsort(X_reg.ravel())
    ax.plot(X_reg.ravel()[sort_idx], snapshot_preds[m][sort_idx],
            color="red", linewidth=2, label=f"Prediction ($F_{{{m}}}$)")

    # Show residuals as thin vertical lines
    for i in range(0, len(X_reg), 5):  # every 5th point to avoid clutter
        ax.plot(
            [X_reg[i, 0], X_reg[i, 0]],
            [snapshot_preds[m][i], y_reg[i]],
            color="blue", alpha=0.2, linewidth=0.7,
        )

    ax.set_title(titles[idx], fontsize=13)
    ax.set_xlabel("x", fontsize=11)
    ax.set_ylabel("y", fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

fig.suptitle("Gradient Boosting: Iteratively Fitting Residuals (learning_rate=0.3)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

Each new tree reduces the residuals. After 10 trees the ensemble closely follows the true sine curve.

---
## Learning Rate in Gradient Boosting

The learning rate $\eta$ controls how much each new tree contributes to the ensemble:

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

- **Small $\eta$** (e.g., 0.01): each tree contributes very little — needs many trees but often generalizes better
- **Large $\eta$** (e.g., 1.0): each tree contributes a lot — fast convergence but higher risk of overfitting

**Key insight:** small $\eta$ + many trees = better generalization (at the cost of more computation).

In [ ]:
np.random.seed(42)

X_reg = np.sort(np.random.uniform(0, 2 * np.pi, 200)).reshape(-1, 1)
y_reg = np.sin(X_reg).ravel() + 0.2 * np.random.randn(200)

learning_rates = [0.01, 0.1, 0.5, 1.0]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

sort_idx = np.argsort(X_reg.ravel())

for i, lr in enumerate(learning_rates):
    ax = axes[i]
    model = GradientBoostingRegressor(
        n_estimators=50, max_depth=2, learning_rate=lr, random_state=42
    )
    model.fit(X_reg, y_reg)
    y_pred = model.predict(X_reg)

    ax.scatter(X_reg, y_reg, alpha=0.3, s=15, color="gray", label="True data")
    ax.plot(X_reg.ravel()[sort_idx], y_pred[sort_idx],
            color="red", linewidth=2, label="GB prediction")
    ax.plot(X_reg.ravel()[sort_idx], np.sin(X_reg.ravel()[sort_idx]),
            color="green", linewidth=1, linestyle="--", alpha=0.7, label="True sin(x)")
    ax.set_title(f"learning_rate = {lr}  (50 trees, depth=2)", fontsize=12)
    ax.set_xlabel("x", fontsize=11)
    ax.set_ylabel("y", fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("Effect of Learning Rate on Gradient Boosting Regression",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

With $\eta = 0.01$ and only 50 trees the model barely fits the data (underfitting). With $\eta = 1.0$ the prediction is noisy and overfits. The sweet spot ($\eta = 0.1$ or $0.5$) gives a smooth fit close to the true function.

---
## Gradient Boosting Classification

Gradient Boosting also works for classification. For binary classification with log-loss, each tree is fitted to the **negative gradient of the log-loss** rather than simple residuals.

Let's see how the decision boundary evolves as we add more trees.

In [ ]:
np.random.seed(42)

X_moons, y_moons = make_moons(n_samples=200, noise=0.3, random_state=42)

n_estimators_list = [1, 10, 50, 200]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for i, n_est in enumerate(n_estimators_list):
    model = GradientBoostingClassifier(
        n_estimators=n_est, max_depth=3, learning_rate=0.1, random_state=42
    )
    model.fit(X_moons, y_moons)
    acc = accuracy_score(y_moons, model.predict(X_moons))
    plot_decision_boundary_2d(
        model, X_moons, y_moons,
        title=f"GradientBoosting — {n_est} estimators (train acc={acc:.2f})",
        ax=axes[i],
    )

plt.tight_layout()
plt.show()

With 1 tree the boundary is crude. With 200 trees the Gradient Boosting classifier captures the non-linear moon shapes accurately.

---
# Part 4: Overfitting in Boosting

Unlike Random Forests, which are relatively resistant to overfitting (thanks to averaging independent trees), **boosting can overfit** if we use too many estimators or a learning rate that is too high.

This is because boosting keeps adding trees that focus on the remaining errors, including **noise** in the training data.

In [ ]:
np.random.seed(42)

X_moons, y_moons = make_moons(n_samples=300, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X_moons, y_moons, test_size=0.3, random_state=42
)

n_range = np.arange(1, 501)
train_acc = []
test_acc = []

for n in n_range:
    model = GradientBoostingClassifier(
        n_estimators=n, learning_rate=0.3, max_depth=5, random_state=42
    )
    model.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, model.predict(X_train)))
    test_acc.append(accuracy_score(y_test, model.predict(X_test)))

plt.figure(figsize=(10, 5))
plt.plot(n_range, train_acc, label="Train accuracy", linewidth=2)
plt.plot(n_range, test_acc, label="Test accuracy", linewidth=2)
plt.xlabel("Number of estimators", fontsize=12)
plt.ylabel("Accuracy", fontsize=12)
plt.title("Gradient Boosting — Overfitting with Too Many Estimators\n(learning_rate=0.3, max_depth=5)",
          fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_idx = np.argmax(test_acc)
print(f"Best test accuracy: {test_acc[best_idx]:.4f} at n_estimators={n_range[best_idx]}")
print(f"Train accuracy at that point: {train_acc[best_idx]:.4f}")

The training accuracy quickly reaches 1.0, but the test accuracy peaks and then starts **decreasing** — a clear sign of overfitting. In practice, you should use **early stopping** or **cross-validation** to select the optimal number of estimators.

---
# Summary

| Aspect | AdaBoost | Gradient Boosting |
|---|---|---|
| **Approach** | Reweights misclassified samples | Fits new trees to residuals (negative gradient) |
| **Loss function** | Exponential loss | Any differentiable loss (MSE, log-loss, etc.) |
| **Weak learner** | Typically decision stumps (depth=1) | Shallow decision trees (depth 3--8) |
| **Key parameters** | `n_estimators`, base estimator | `n_estimators`, `learning_rate`, `max_depth` |
| **Overfitting risk** | Moderate — can overfit with noisy data | Higher — mitigated by small learning rate + early stopping |

### Key Takeaways

1. **Boosting is sequential**: each new model corrects the errors of the ensemble so far.
2. **AdaBoost** increases the weight of misclassified samples; the final prediction is a weighted vote.
3. **Gradient Boosting** fits each new tree to the residuals (negative gradient of the loss). This is gradient descent in function space.
4. The **learning rate** $\eta$ controls the contribution of each tree. Smaller values require more trees but generalize better.
5. Unlike bagging, boosting **can overfit** — always monitor validation performance.